In [5]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor

from xrfm import xRFM

c:\Users\atgee\OneDrive\COMP9417\COMP9417\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def prepare_tabular_data(df, target_col, drop_cols=None, test_size=0.2, val_size=0.2, random_state=42):
    if drop_cols is None:
        drop_cols = []

    # Separate X and y
    y = df[target_col].to_numpy()
    X = df.drop(columns=[target_col] + drop_cols)

    # Identify column types
    num_cols = X.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = X.select_dtypes(include=["object"]).columns

    # Preprocessing pipelines
    num_pipeline = [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]

    cat_pipeline = [
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(num_pipeline), num_cols),
            ("cat", Pipeline(cat_pipeline), cat_cols)
        ]
    )

    # Split: train/test
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y if len(np.unique(y)) < 10 else None
    )

    # Split: train/val
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=val_size, random_state=random_state,
        stratify=y_train_full if len(np.unique(y)) < 10 else None
    )

    # Fit on train only
    X_train_proc = preprocessor.fit_transform(X_train)
    X_val_proc = preprocessor.transform(X_val)
    X_test_proc = preprocessor.transform(X_test)

    # Convert to dense (important for xRFM)
    X_train_proc = X_train_proc.toarray() if hasattr(X_train_proc, "toarray") else X_train_proc
    X_val_proc = X_val_proc.toarray() if hasattr(X_val_proc, "toarray") else X_val_proc
    X_test_proc = X_test_proc.toarray() if hasattr(X_test_proc, "toarray") else X_test_proc

    # Force numeric dtype
    X_train_proc = X_train_proc.astype("float32")
    X_val_proc = X_val_proc.astype("float32")
    X_test_proc = X_test_proc.astype("float32")

    return X_train_proc, X_val_proc, X_test_proc, y_train, y_val, y_test, preprocessor

In [7]:
hc_df = pd.read_csv("../datasets/healthcare-dataset-stroke-data.csv")
hc_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   str    
 6   work_type          5110 non-null   str    
 7   Residence_type     5110 non-null   str    
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   str    
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), str(5)
memory usage: 479.2 KB


In [8]:
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor = prepare_tabular_data(
    hc_df,
    target_col="stroke",
    drop_cols=["id"]
)

C:\Users\atgee\AppData\Local\Temp\ipykernel_44580\3190032777.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns


In [9]:
np.random.seed(0)
DEVICE = torch.device("cuda")
bw = 5.
reg = 1e-3
iters = 5
max_leaf_size = 55_000

xrfm_params = {
    'model': {
        'kernel': "sum_power_laplace",
        'bandwidth': bw,
        'exponent': 1.0,
        'diag': False,
        'bandwidth_mode': "constant"
    },
    'fit': {
        'reg': reg,
        'iters': iters,
        'verbose': True,
        'early_stop_rfm': True,
    }
}
default_rfm_params = {
    'model': {
        "kernel": 'l2_high_dim',
        "exponent": 1.0,
        "bandwidth": 10.0,
        "diag": False,
        "bandwidth_mode": "constant"
    },
    'fit' : {
        "get_agop_best_model": True,
        "return_best_params": False,
        "reg": 1e-3,
        "iters": 0,
        "early_stop_rfm": False,
        "verbose": False
    }
}

In [10]:
xrfm_model = xRFM()

xrfm_model.fit(X_train, y_train, X_val, y_val)
y_pred = xrfm_model.predict(X_test)

None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 3270, d: 21, and nval: 818
Using cheap batch size
Optimal M batch size: 3270
Time taken for round 0: 0.07681918144226074 seconds
Using cheap batch size
Optimal M batch size: 3270
Time taken for round 1: 0.06514477729797363 seconds
Using cheap batch size
Optimal M batch size: 3270
Time taken for round 2: 0.06451559066772461 seconds


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Using cheap batch size
Optimal M batch size: 3270
Time taken for round 3: 0.06692171096801758 seconds
Using cheap batch size
Optimal M batch size: 3270
Time taken for round 4: 0.06234335899353027 seconds
Using cheap batch size
Optimal M batch size: 3270
Tree has no split, stopping training
Using hard routing for tree prediction


In [11]:
accuracy = np.sum(y_pred == y_test)/len(y_test)
print(accuracy)

0.9315068493150684
